# Notebook 02 — Limpeza e Pré-processamento dos Dados

**Projeto Mensal 3 — Tratamento de Dados**  
**Base:** Cargos Vagos e Vacâncias do Poder Executivo Federal Civil — competência 03/2026

---

## Objetivo deste notebook

Aplicar o **tratamento dos dados** identificados como problemáticos no Notebook 01, gerando uma base intermediária limpa e padronizada. As novas variáveis derivadas (feature engineering) serão criadas no Notebook 03.

## Etapas cobertas

Este notebook cobre as seguintes etapas do enunciado:

| Etapa | Descrição |
|---|---|
| **6.6** | Seleção de dados |
| **6.7** | Limpeza e pré-processamento |
| **6.8** | Tratamento de valores faltantes |
| **6.9** | Tratamento de outliers |
| **6.10** | Transformação dos dados (correções estruturais) |

A **Análise Exploratória de Dados (etapa 6.5)** será feita no Notebook 02b (separado), após a limpeza concluída.

## Insumo de entrada

- `dados_brutos/CargosVagosVacancias_202603.csv` — base original (preservada e imutável)

## Produto de saída

- `dados_tratados/dataset_pos_limpeza.csv` — base limpa, com tipos corretos, sem nulos e com flag de outliers

## Setup — Bibliotecas e configurações

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Configurações de exibição
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)

# Caminhos do projeto
BASE_DIR = Path('..').resolve()
DADOS_BRUTOS = BASE_DIR / 'dados_brutos'
DADOS_TRATADOS = BASE_DIR / 'dados_tratados'

print(f"Base do projeto: {BASE_DIR}")
print(f"Versão pandas:   {pd.__version__}")
print(f"Versão numpy:    {np.__version__}")

Base do projeto: /home/claude/PM3_CargosVagosVacancias
Versão pandas:   3.0.2
Versão numpy:    2.4.4


## Carga da base bruta

In [2]:
ARQUIVO_BRUTO = DADOS_BRUTOS / 'CargosVagosVacancias_202603.csv'
df = pd.read_csv(ARQUIVO_BRUTO)

print(f"Base bruta carregada:")
print(f"  Linhas:  {df.shape[0]:,}".replace(',', '.'))
print(f"  Colunas: {df.shape[1]}")

# Snapshot do shape inicial para comparação no final
SHAPE_INICIAL = df.shape

Base bruta carregada:
  Linhas:  12.769
  Colunas: 20


## 1. Seleção dos dados (etapa 6.6)

> *"Você deverá escolher quais colunas e registros serão mantidos para o dataset final. A seleção precisa ter justificativa técnica."*

### 1.1 Justificativa de manutenção das colunas

Após o diagnóstico do Notebook 01, decidimos **manter todas as 20 colunas** da base bruta. A análise mostrou que cada coluna apresenta valor analítico:

| Coluna | Justificativa de manutenção |
|---|---|
| `ANO_MES` | Marca temporal do snapshot — essencial para análise histórica |
| `ORGAO`, `NOME_ORGAO`, `SIGLA_ORGAO` | Identificação do órgão (chave + descritivos) |
| `NIVEL` | Variável categórica analítica (escolaridade do cargo) |
| `CARGO`, `NOME_CARGO`, `PLANO_CARREIRA` | Identificação do cargo e estrutura de carreira |
| `APROVADA`, `DISTRIBUIDA`, `OCUPADA`, `VAGA` | Métricas principais do quadro de pessoal |
| 7 colunas `VACANCIA_POR_*` | Discriminação dos tipos de vacância — relevante para análise de gestão |
| `CARGO_EM_EXTINCAO` | Indicador binário de descontinuidade |

**Nenhuma coluna será removida** — todas se mostraram úteis para análise.

### 1.2 Justificativa de manutenção dos registros

Verificamos a existência de registros candidatos a remoção:

In [3]:
# Linhas onde TODAS as variáveis principais são zero (registros "vazios")
mask_tudo_zero = (
    (df['APROVADA'] == 0) &
    (df['DISTRIBUIDA'] == 0) &
    (df['OCUPADA'] == 0) &
    (df['VAGA'] == 0)
)
print(f"Linhas com APROVADA=DISTRIBUIDA=OCUPADA=VAGA = 0: {mask_tudo_zero.sum()}")

# Linhas onde APROVADA = 0 (cargo sem aprovação legal)
print(f"Linhas com APROVADA = 0: {(df['APROVADA'] == 0).sum()}")

# Linhas sem nenhuma vacância no mês
cols_vac = [c for c in df.columns if c.startswith('VACANCIA_')]
sem_vacancia = (df[cols_vac].sum(axis=1) == 0)
print(f"Linhas sem nenhuma vacância no mês: {sem_vacancia.sum():,} ({sem_vacancia.sum()/len(df)*100:.1f}%)".replace(',', '.'))

Linhas com APROVADA=DISTRIBUIDA=OCUPADA=VAGA = 0: 0
Linhas com APROVADA = 0: 0
Linhas sem nenhuma vacância no mês: 7.403 (58.0%)


**Conclusão da etapa 6.6:**

- Não há registros "vazios" na base — todas as 12.769 linhas representam combinações reais (órgão, cargo) com pelo menos uma vaga aprovada.
- As linhas sem vacância no mês (58%) **não devem ser removidas** — vacância é um evento raro, e a ausência dela em um mês específico é informação válida (representa estabilidade do cargo no período).

**Decisão final:** mantemos **todas as 20 colunas e todos os 12.769 registros**.

## 2. Limpeza e pré-processamento (etapa 6.7)

Aplicação das correções identificadas no Notebook 01: renomeação de colunas, conversão de tipos e padronização.

### 2.1 Renomeação de colunas para snake_case

Problema **P10** do diagnóstico: colunas em MAIÚSCULAS, fora do padrão snake_case da comunidade Python.

In [4]:
# Mapeamento explícito (mais legível que aplicar str.lower() em tudo)
renomear = {
    'ANO_MES': 'ano_mes',
    'ORGAO': 'cod_orgao',
    'NOME_ORGAO': 'nome_orgao',
    'SIGLA_ORGAO': 'sigla_orgao',
    'NIVEL': 'nivel',
    'CARGO': 'cod_cargo',
    'PLANO_CARREIRA': 'plano_carreira',
    'NOME_CARGO': 'nome_cargo',
    'APROVADA': 'qtd_aprovada',
    'DISTRIBUIDA': 'qtd_distribuida',
    'OCUPADA': 'qtd_ocupada',
    'VAGA': 'qtd_vaga',
    'VACANCIA_POR_EXONERACAO': 'vac_exoneracao',
    'VACANCIA_POR_DEMISSAO': 'vac_demissao',
    'VACANCIA_POR_PROMOCAO': 'vac_promocao',
    'VACANCIA_POR_READAPTACAO': 'vac_readaptacao',
    'VACANCIA_POR_APOSENTADORIA': 'vac_aposentadoria',
    'VACANCIA_POR_POSSE_CARGO_INAC': 'vac_posse_cargo_inac',
    'VACANCIA_POR_FALECIMENTO': 'vac_falecimento',
    'CARGO_EM_EXTINCAO': 'em_extincao',
}

df = df.rename(columns=renomear)
print("Colunas após renomeação:")
for c in df.columns:
    print(f"  - {c}")

Colunas após renomeação:
  - ano_mes
  - cod_orgao
  - nome_orgao
  - sigla_orgao
  - nivel
  - cod_cargo
  - plano_carreira
  - nome_cargo
  - qtd_aprovada
  - qtd_distribuida
  - qtd_ocupada
  - qtd_vaga
  - vac_exoneracao
  - vac_demissao
  - vac_promocao
  - vac_readaptacao
  - vac_aposentadoria
  - vac_posse_cargo_inac
  - vac_falecimento
  - em_extincao


**Observações sobre a renomeação:**

- `ORGAO` → `cod_orgao` e `CARGO` → `cod_cargo`: o prefixo `cod_` deixa explícito que são códigos identificadores, não valores numéricos.
- `APROVADA`, `DISTRIBUIDA`, `OCUPADA`, `VAGA` → `qtd_*`: o prefixo `qtd_` indica claramente que são quantidades (vagas).
- `VACANCIA_POR_*` → `vac_*`: redução do tamanho do nome sem perda de clareza.
- `CARGO_EM_EXTINCAO` → `em_extincao`: nome mais direto.

### 2.2 Conversão de tipos — códigos identificadores

Problemas **P04** e **P05**: `cod_orgao` e `cod_cargo` estão como `int64`, mas são identificadores. Convertendo para string e preservando zeros à esquerda.

In [5]:
# Códigos governamentais brasileiros têm largura fixa:
# - ORGAO: 5 dígitos (sistema SIORG/SIAPE)
# - CARGO: 6 dígitos (3 primeiros = plano/carreira, 3 últimos = cargo interno)
# Conforme o dicionário oficial da base.

df['cod_orgao'] = df['cod_orgao'].astype(str).str.zfill(5)
df['cod_cargo'] = df['cod_cargo'].astype(str).str.zfill(6)

print("Amostras após conversão:")
print(df[['cod_orgao', 'cod_cargo']].drop_duplicates().head(10).to_string(index=False))
print(f"\nTipo cod_orgao: {df['cod_orgao'].dtype}")
print(f"Tipo cod_cargo: {df['cod_cargo'].dtype}")

Amostras após conversão:
cod_orgao cod_cargo
    13000    008001
    13000    009001
    13000    010057
    13000    015001
    13000    104001
    13000    104002
    13000    409010
    13000    480246
    13000    481004
    13000    830001

Tipo cod_orgao: str
Tipo cod_cargo: str


### 2.3 Conversão de tipos — data

Problema **P03**: `ano_mes` está como `int64` (formato AAAAMM), mas é semanticamente uma data.

In [6]:
# Converter ANO_MES (int 202603) para datetime (2026-03-01, primeiro dia do mês)
df['ano_mes'] = pd.to_datetime(df['ano_mes'].astype(str) + '01', format='%Y%m%d')

print(f"Tipo da coluna ano_mes: {df['ano_mes'].dtype}")
print(f"Valores únicos:  {df['ano_mes'].unique()}")
print(f"\nAmostra:")
print(df[['ano_mes', 'cod_orgao', 'cod_cargo']].head(3).to_string(index=False))

Tipo da coluna ano_mes: datetime64[us]
Valores únicos:  <DatetimeArray>
['2026-03-01 00:00:00']
Length: 1, dtype: datetime64[us]

Amostra:
   ano_mes cod_orgao cod_cargo
2026-03-01     13000    008001
2026-03-01     13000    009001
2026-03-01     13000    010057


### 2.4 Conversão de tipos — variáveis categóricas

Variáveis categóricas se beneficiam do tipo `category` do pandas: menos memória e operações vetoriais mais rápidas em agrupamentos.

In [7]:
# Antes da conversão para category, vamos tratar os nulos (próxima seção 6.8).
# Se converter agora, os nulos atrapalham a definição das categorias.
# Por isso a conversão para category fica para o final desta seção.

# Memória antes das conversões:
mem_antes = df.memory_usage(deep=True).sum() / 1024**2
print(f"Memória atual: {mem_antes:.2f} MB")

# Validação intermediária
print(f"\nTipos atuais das colunas:")
df.dtypes

Memória atual: 7.74 MB

Tipos atuais das colunas:


ano_mes                 datetime64[us]
cod_orgao                          str
nome_orgao                         str
sigla_orgao                        str
nivel                              str
cod_cargo                          str
plano_carreira                     str
nome_cargo                         str
qtd_aprovada                     int64
qtd_distribuida                  int64
qtd_ocupada                      int64
qtd_vaga                         int64
vac_exoneracao                   int64
vac_demissao                     int64
vac_promocao                     int64
vac_readaptacao                  int64
vac_aposentadoria                int64
vac_posse_cargo_inac             int64
vac_falecimento                  int64
em_extincao                        str
dtype: object

## 3. Tratamento de valores faltantes (etapa 6.8)

Problemas **P01** (NIVEL com 1.255 nulos) e **P02** (PLANO_CARREIRA com 363 nulos).

### 3.1 Diagnóstico antes do tratamento

In [8]:
nulos_antes = df.isnull().sum()
print("Nulos antes do tratamento:")
print(nulos_antes[nulos_antes > 0])

Nulos antes do tratamento:
nivel             1255
plano_carreira     363
dtype: int64


### 3.2 Estratégia de tratamento

Para essas duas variáveis, optamos por **preencher com a categoria `"NAO_INFORMADO"`**, pelos seguintes motivos técnicos:

**Por que NÃO usar a moda (categoria mais frequente)?**  
Imputar "NS" (Nível Superior) ou "PLANO_CARGOS_TAE" (PCCTAE) — que são as modas — distorceria a análise. O fato de o nível ou o plano de carreira não ter sido informado **é em si uma informação**: pode indicar cargos legados, em extinção ou em situação administrativa especial.

**Por que NÃO remover essas linhas?**  
São 1.618 linhas no total (≈ 12,7% da base). Removê-las descartaria informação válida das outras 18 colunas que estão completas para esses registros.

**Por que `"NAO_INFORMADO"` é adequado:**  
- Preserva 100% dos registros.
- Torna explícito que o dado faltou na fonte (vs. dado preenchido errado).
- Permite agrupar e analisar esses casos separadamente, se desejado.
- É prática recomendada em datasets administrativos governamentais brasileiros, onde o preenchimento de alguns campos não é obrigatório.

In [9]:
# Aplicar a estratégia em NIVEL
df['nivel'] = df['nivel'].fillna('NAO_INFORMADO')

# Aplicar a estratégia em PLANO_CARREIRA
df['plano_carreira'] = df['plano_carreira'].fillna('NAO_INFORMADO')

# Verificação
print("Nulos depois do tratamento:")
nulos_depois = df.isnull().sum()
print(nulos_depois[nulos_depois > 0] if nulos_depois.sum() > 0 else "  Nenhum nulo restante")

print(f"\nDistribuição final de 'nivel':")
print(df['nivel'].value_counts())

print(f"\nDistribuição final de 'plano_carreira' (top 5):")
print(df['plano_carreira'].value_counts().head(5))

Nulos depois do tratamento:
  Nenhum nulo restante

Distribuição final de 'nivel':
nivel
NS               5792
NI               5712
NAO_INFORMADO    1255
NM                 10
Name: count, dtype: int64

Distribuição final de 'plano_carreira' (top 5):
plano_carreira
Plano de Carreiras dos Cargos Técnico-Administrativos em Educação - PCCTAE    8391
Plano Geral de Cargos do Poder Executivo - PGPE                               1197
Carreira da Previdência, da Saúde e do Trabalho                                418
NAO_INFORMADO                                                                  363
Plano Especial de Cargos da Cultura                                            348
Name: count, dtype: int64


### 3.3 Conversão final para tipo `category`

Agora que não há mais nulos nas variáveis categóricas, é seguro converter para o tipo otimizado.

In [10]:
# Converter variáveis categóricas
df['nivel'] = df['nivel'].astype('category')
df['plano_carreira'] = df['plano_carreira'].astype('category')
df['em_extincao'] = df['em_extincao'].astype('category')

# Validação
mem_depois = df.memory_usage(deep=True).sum() / 1024**2
economia = ((mem_antes - mem_depois) / mem_antes) * 100

print(f"Memória depois: {mem_depois:.2f} MB")
print(f"Economia: {economia:.1f}%")

print(f"\nTipos finais das colunas:")
df.dtypes

Memória depois: 5.06 MB
Economia: 34.6%

Tipos finais das colunas:


ano_mes                 datetime64[us]
cod_orgao                          str
nome_orgao                         str
sigla_orgao                        str
nivel                         category
cod_cargo                          str
plano_carreira                category
nome_cargo                         str
qtd_aprovada                     int64
qtd_distribuida                  int64
qtd_ocupada                      int64
qtd_vaga                         int64
vac_exoneracao                   int64
vac_demissao                     int64
vac_promocao                     int64
vac_readaptacao                  int64
vac_aposentadoria                int64
vac_posse_cargo_inac             int64
vac_falecimento                  int64
em_extincao                   category
dtype: object

## 4. Tratamento de outliers (etapa 6.9)

Problema **P09**: distribuições altamente assimétricas em `qtd_aprovada`, `qtd_distribuida`, `qtd_ocupada`, `qtd_vaga`.

### 4.1 Estratégia adotada

Conforme decidido no Notebook 01, **os outliers serão preservados** porque representam casos reais e legítimos (grandes órgãos institucionais: INSS, universidades federais, Polícia Federal). Removê-los descartaria justamente os dados mais relevantes para análise de gestão de pessoas no Executivo Federal.

A estratégia será:

1. Identificar outliers em `qtd_aprovada` pelo método IQR (Intervalo Interquartil).
2. Criar uma **coluna indicadora binária `is_outlier`** marcando esses registros.
3. Permitir, em análises futuras, segmentar entre "cargos típicos" e "cargos institucionais grandes".

### 4.2 Identificação e marcação

In [11]:
# Cálculo dos limites IQR para qtd_aprovada
Q1 = df['qtd_aprovada'].quantile(0.25)
Q3 = df['qtd_aprovada'].quantile(0.75)
IQR = Q3 - Q1
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print(f"Cálculo dos limites IQR para qtd_aprovada:")
print(f"  Q1 (25º percentil): {Q1}")
print(f"  Q3 (75º percentil): {Q3}")
print(f"  IQR (Q3 - Q1):      {IQR}")
print(f"  Limite inferior:    {limite_inferior}")
print(f"  Limite superior:    {limite_superior}")

Cálculo dos limites IQR para qtd_aprovada:
  Q1 (25º percentil): 1.0
  Q3 (75º percentil): 12.0
  IQR (Q3 - Q1):      11.0
  Limite inferior:    -15.5
  Limite superior:    28.5


In [12]:
# Criar a coluna indicadora
df['is_outlier'] = (
    (df['qtd_aprovada'] < limite_inferior) |
    (df['qtd_aprovada'] > limite_superior)
).astype(int)

# Resumo
total_outliers = df['is_outlier'].sum()
print(f"Registros marcados como outliers: {total_outliers:,} ({total_outliers/len(df)*100:.1f}%)".replace(',', '.'))
print(f"Registros típicos:                {len(df) - total_outliers:,}".replace(',', '.'))

Registros marcados como outliers: 1.879 (14.7%)
Registros típicos:                10.890


In [13]:
# Validar visualizando os 5 maiores outliers
print("5 maiores outliers (cargos com mais vagas aprovadas):\n")
print(df[df['is_outlier'] == 1].nlargest(5, 'qtd_aprovada')[
    ['nome_orgao', 'nome_cargo', 'qtd_aprovada', 'qtd_ocupada', 'qtd_vaga']
].to_string(index=False))

5 maiores outliers (cargos com mais vagas aprovadas):



                          nome_orgao                               nome_cargo  qtd_aprovada  qtd_ocupada  qtd_vaga
 INSTITUTO NACIONAL DE SEGURO SOCIAL                 TECNICO DO SEGURO SOCIAL         34209        13302     20907
                 MINISTERIO DA SAUDE                                   MEDICO         25473         2562     22911
               MINISTERIO DA FAZENDA AUDITOR-FISCAL DA RECEITA FEDERAL BRASIL         19642         6937     12705
               MINISTERIO DA FAZENDA   ANALISTA TRIBUTARIO REC FEDERAL BRASIL         16341         6293     10048
DEPTO. DE POLICIA RODOVIARIA FEDERAL              POLICIAL RODOVIARIO FEDERAL         13097        12840       257


**Interpretação dos outliers:**

Os outliers correspondem a cargos institucionais legítimos (Polícia Federal, Receita Federal, universidades federais, INSS). **Manter esses registros é essencial** — eles representam a estrutura mais relevante do Executivo Federal em termos de quantitativo de servidores.

A coluna `is_outlier` permitirá:
- Análises segmentadas entre cargos típicos e institucionais.
- Aplicação cuidadosa de técnicas de normalização no Notebook 03 (alguns métodos como Min-Max são sensíveis a outliers).
- Documentação transparente da decisão técnica.

## 5. Transformação dos dados (etapa 6.10)

> *"Você deverá transformar os dados para deixá-los mais adequados à análise."*

Esta seção consolida as **transformações estruturais** já realizadas anteriormente e que se enquadram explicitamente nesta etapa:

| Transformação | Origem | Resultado | Onde foi feita |
|---|---|---|---|
| Conversão de texto/int para data | `ano_mes` int64 (202603) | datetime (2026-03-01) | Seção 2.3 |
| Conversão de identificadores numéricos para texto | `cod_orgao`, `cod_cargo` int64 | string com largura fixa | Seção 2.2 |
| Padronização de nomes de colunas | MAIÚSCULAS | snake_case | Seção 2.1 |
| Padronização de variáveis categóricas | object com nulos | category sem nulos | Seções 3.2 e 3.3 |
| Criação de indicador binário | Análise IQR | `is_outlier` (0/1) | Seção 4.2 |

**Transformações adicionais** — criação de novas variáveis derivadas (ano, mês, total de vacâncias, taxas de ocupação, decomposição do código do cargo, etc.) — serão realizadas no **Notebook 03 (Feature Engineering)**, mantendo a separação conceitual entre "corrigir/padronizar dados" (etapa atual) e "criar novas informações" (próxima etapa).

## 6. Validação final e exportação

### 6.1 Comparação antes vs. depois

In [14]:
print("=== ANTES (base bruta) ===")
print(f"  Linhas:  {SHAPE_INICIAL[0]:,}".replace(',', '.'))
print(f"  Colunas: {SHAPE_INICIAL[1]}")
print(f"  Memória: {mem_antes:.2f} MB")
print(f"  Nulos:   {nulos_antes.sum():,}".replace(',', '.'))

print(f"\n=== DEPOIS (pós-limpeza) ===")
print(f"  Linhas:  {df.shape[0]:,}".replace(',', '.'))
print(f"  Colunas: {df.shape[1]} (+ {df.shape[1] - SHAPE_INICIAL[1]} nova coluna 'is_outlier')")
print(f"  Memória: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"  Nulos:   {df.isnull().sum().sum()}")

=== ANTES (base bruta) ===
  Linhas:  12.769
  Colunas: 20
  Memória: 7.74 MB
  Nulos:   1.618

=== DEPOIS (pós-limpeza) ===
  Linhas:  12.769
  Colunas: 21 (+ 1 nova coluna 'is_outlier')
  Memória: 5.16 MB
  Nulos:   0


In [15]:
# Visualização final do DataFrame
print("Primeiras linhas do dataset pós-limpeza:\n")
df.head(5)

Primeiras linhas do dataset pós-limpeza:



,ano_mes,cod_orgao,nome_orgao,sigla_orgao,nivel,cod_cargo,plano_carreira,nome_cargo,qtd_aprovada,qtd_distribuida,qtd_ocupada,qtd_vaga,vac_exoneracao,vac_demissao,vac_promocao,vac_readaptacao,vac_aposentadoria,vac_posse_cargo_inac,vac_falecimento,em_extincao,is_outlier
0,2026-03-01,13000,"MINIST.DA AGRICULTURA,PECUARIA E ABAST.",MAPA,NI,008001,Plano de Classificação de Cargos - PCC,AGENTE ADMINISTRATIVO,2,2,2,0,14,1,0,0,123,4,0,S,0
1,2026-03-01,13000,"MINIST.DA AGRICULTURA,PECUARIA E ABAST.",MAPA,NS,009001,Plano de Classificação de Cargos - PCC,MEDICO,1,1,1,0,0,0,0,0,2,0,0,S,0
2,2026-03-01,13000,"MINIST.DA AGRICULTURA,PECUARIA E ABAST.",MAPA,NAO_INFORMADO,010057,Plano de Classificação de Cargos - PCC,AUXILIAR OPERACIONAL DE AGROPECUARIA,1,1,1,0,1,0,0,0,1,0,0,S,0
3,2026-03-01,13000,"MINIST.DA AGRICULTURA,PECUARIA E ABAST.",MAPA,NS,015001,Plano de Classificação de Cargos - PCC,TECNICO DE PLANEJAMENTO,1,1,1,0,0,0,0,0,0,0,0,S,0
4,2026-03-01,13000,"MINIST.DA AGRICULTURA,PECUARIA E ABAST.",MAPA,NI,104001,Cargos de Atividades Técnicas de Fiscalização Agropecuária do Quadro de Pess...,AGENTE DE INSP SANIT IND PROD ORIGEM ANI,1,1,0,1,0,0,0,0,2,0,0,N,0


In [16]:
# Estrutura final
print("Tipos das colunas no dataset pós-limpeza:\n")
df.info()

Tipos das colunas no dataset pós-limpeza:



<class 'pandas.DataFrame'>
RangeIndex: 12769 entries, 0 to 12768
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   ano_mes               12769 non-null  datetime64[us]
 1   cod_orgao             12769 non-null  str           
 2   nome_orgao            12769 non-null  str           
 3   sigla_orgao           12769 non-null  str           
 4   nivel                 12769 non-null  category      
 5   cod_cargo             12769 non-null  str           
 6   plano_carreira        12769 non-null  category      
 7   nome_cargo            12769 non-null  str           
 8   qtd_aprovada          12769 non-null  int64         
 9   qtd_distribuida       12769 non-null  int64         
 10  qtd_ocupada           12769 non-null  int64         
 11  qtd_vaga              12769 non-null  int64         
 12  vac_exoneracao        12769 non-null  int64         
 13  vac_demissao          12769

### 6.2 Exportação do dataset intermediário

Este não é o dataset final do projeto (que vem no Notebook 04), mas um **ponto de checkpoint** após a limpeza. Os próximos notebooks partirão dele.

**Atenção técnica:** como CSV não preserva tipos de dados, ao reler o arquivo precisamos especificar explicitamente que `cod_orgao` e `cod_cargo` são strings (caso contrário, os zeros à esquerda do código do cargo seriam perdidos). A célula abaixo demonstra a especificação correta de releitura, que será usada nos próximos notebooks.

In [17]:
# Garantir que a pasta de saída existe
DADOS_TRATADOS.mkdir(parents=True, exist_ok=True)

caminho_saida = DADOS_TRATADOS / 'dataset_pos_limpeza.csv'
df.to_csv(caminho_saida, index=False, encoding='utf-8')

# IMPORTANTE: ao reler o CSV, é necessário forçar o dtype dos códigos
# identificadores como string, senão o pandas detecta como int e os zeros
# à esquerda são perdidos. Os próximos notebooks usarão esta especificação.
DTYPES_LEITURA = {
    'cod_orgao': str,
    'cod_cargo': str,
    'em_extincao': 'category',
}

# Validação: ler de volta com os dtypes corretos e conferir
df_validacao = pd.read_csv(
    caminho_saida,
    dtype=DTYPES_LEITURA,
    parse_dates=['ano_mes']
)

print(f"Arquivo salvo: {caminho_saida}")
print(f"Tamanho:       {caminho_saida.stat().st_size / 1024:.1f} KB")
print(f"\nValidação de releitura:")
print(f"  Linhas lidas:  {len(df_validacao):,}".replace(',', '.'))
print(f"  Colunas lidas: {df_validacao.shape[1]}")
print(f"  cod_orgao tem largura {df_validacao['cod_orgao'].str.len().min()}-{df_validacao['cod_orgao'].str.len().max()} chars")
print(f"  cod_cargo tem largura {df_validacao['cod_cargo'].str.len().min()}-{df_validacao['cod_cargo'].str.len().max()} chars")
print(f"  ano_mes dtype: {df_validacao['ano_mes'].dtype}")

Arquivo salvo: /home/claude/PM3_CargosVagosVacancias/dados_tratados/dataset_pos_limpeza.csv
Tamanho:       2308.7 KB

Validação de releitura:
  Linhas lidas:  12.769
  Colunas lidas: 21
  cod_orgao tem largura 5-5 chars
  cod_cargo tem largura 6-6 chars
  ano_mes dtype: datetime64[us]


## 7. Conclusão

### O que foi feito neste notebook

1. **Etapa 6.6 (Seleção):** mantivemos todas as 20 colunas e os 12.769 registros, com justificativa técnica.
2. **Etapa 6.7 (Limpeza):**
   - Renomeamos as 20 colunas para `snake_case`.
   - Convertemos `cod_orgao` e `cod_cargo` para strings com largura fixa (preservando zeros à esquerda).
   - Convertemos `ano_mes` de inteiro para `datetime`.
   - Convertemos `nivel`, `plano_carreira` e `em_extincao` para o tipo `category`.
3. **Etapa 6.8 (Valores faltantes):** preenchemos 1.255 nulos em `nivel` e 363 em `plano_carreira` com a categoria `"NAO_INFORMADO"`, preservando 100% dos registros.
4. **Etapa 6.9 (Outliers):** identificamos outliers em `qtd_aprovada` pelo método IQR e os marcamos com a coluna binária `is_outlier`, sem remoção de dados.
5. **Etapa 6.10 (Transformações):** consolidamos as transformações estruturais aplicadas (conversões de tipo, padronização de nomes, criação de indicador).

### O que vem agora

- **Notebook 02b — AED (etapa 6.5):** análise exploratória completa do dataset pós-limpeza, com os gráficos exigidos pelo enunciado.
- **Notebook 03 — Feature Engineering (etapas 6.11 a 6.14):** criação de novas variáveis derivadas, agregações, normalização e discretização.
- **Notebook 04 — Dataset Final (etapas 6.15 e 6.16):** consolidação e geração do catálogo de dados.

---
*Fim do Notebook 02 — Limpeza e Pré-processamento*